![Banner](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/datacrunch-2/assets/banner.webp)

# DataCrunch 2 — Transformer, warm-started + incrementally fine-tuned

**Companion notebook required first:** run `pretrain_transformer.ipynb` on
your own GPU, then upload the `transformer_checkpoint.pt` it produces as a
resource alongside *this* notebook when you submit.

**Why this design:** CrunchDAO's runtime is CPU-only and calls `train()` 9
times inside a shared weekly compute quota (this is what killed the two
previous stacking-ensemble attempts). Training a transformer from scratch
on CPU, 9 times, doesn't fit that budget either. So the expensive part —
full pretraining — happens offline, where you have GPU compute and no time
pressure. Each CrunchDAO-side `train()` call then does a **small,
cheap fine-tune**: a couple epochs, low learning rate, only on the most
recent `FINETUNE_WINDOW_MOONS` moons rather than the whole growing
history. That's the mechanism that keeps every one of the 9 calls fast.

**⚠️ `FTTransformerRegressor` below must stay byte-for-byte identical to
the copy in `pretrain_transformer.ipynb`** — the checkpoint is a
`state_dict`, which only loads into a matching architecture.

**Please verify the resource actually lands where `train()` expects it.**
I'm inferring the upload mechanism from CrunchDAO's docs, not from a run I
can see myself — after submitting, check the run log for a line
confirming `transformer_checkpoint.pt` was found (similar to how
`model.joblib` showed up in your earlier successful run). The code below
prints clearly whether it warm-started or is cold-starting, specifically
so this is easy to catch if it goes wrong.


In [ ]:
# CPU wheel specifically -- avoids pulling a large CUDA build onto a CPU-only runtime
%pip install crunch-cli --upgrade --quiet --progress-bar off
%pip install torch --index-url https://download.pytorch.org/whl/cpu --upgrade --quiet

# Setup your local environment
!crunch setup-notebook datacrunch-2 0JiCmmP21Ca88X8TApRuDHMH

## Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
print("Libraries loaded.")

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Config & helpers

In [ ]:
# @crunch/keep:on
RANDOM_STATE = 0
ID_COLUMNS = ["id", "moon"]
CHECKPOINT_FILENAME = "transformer_checkpoint.pt"  # must match the filename you uploaded as a resource
# @crunch/keep:off

# Only affects THIS notebook's local re-runs, not the cloud submission (this
# line is a plain top-level statement, so the processor strips it) -- fine,
# since determinism only matters for local dev here.
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def get_feature_columns(df: pd.DataFrame):
    """All Feature_* columns — explicitly excludes id/moon/target so they
    never get fed into the model as if they were predictive features."""
    return [c for c in df.columns if c not in ID_COLUMNS and c != "target"]


def spearman(y_true, y_pred) -> float:
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(corr) else corr

## Model

Must match `pretrain_transformer.ipynb` exactly — see the warning above.

In [ ]:
class FTTransformerRegressor(nn.Module):
    def __init__(self, n_features: int, n_bins: int = 7, d_model: int = 64,
                 n_heads: int = 4, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.n_features = n_features
        self.value_embedding = nn.Embedding(n_bins, d_model)
        self.feature_id_embedding = nn.Embedding(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_bins: torch.Tensor) -> torch.Tensor:
        batch_size = x_bins.shape[0]
        feature_ids = torch.arange(self.n_features, device=x_bins.device).unsqueeze(0)
        tokens = self.value_embedding(x_bins) + self.feature_id_embedding(feature_ids)
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = encoded[:, 0, :]
        return self.head(cls_out).squeeze(-1)


def build_model(config: dict) -> FTTransformerRegressor:
    return FTTransformerRegressor(
        n_features=config["n_features"],
        n_bins=config.get("n_bins", 7),
        d_model=config.get("d_model", 64),
        n_heads=config.get("n_heads", 4),
        n_layers=config.get("n_layers", 3),
        dropout=config.get("dropout", 0.1),
    )


def save_checkpoint(path, model, config, feature_columns, target_mean, target_std):
    torch.save({
        "model_state": model.state_dict(),
        "config": config,
        "feature_columns": feature_columns,
        "target_mean": target_mean,
        "target_std": target_std,
    }, path)


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = build_model(checkpoint["config"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    return model, checkpoint

## Fine-tuning config

`DEFAULT_MODEL_CONFIG` is only a fallback for a cold start (no checkpoint
found) — when warm-starting, the architecture actually used comes from the
checkpoint itself, so pretraining and fine-tuning always agree even if you
change `PRETRAIN_CONFIG` later without updating this cell.

In [ ]:
# @crunch/keep:on
FINETUNE_WINDOW_MOONS = 60   # only fine-tune on the most recent N moons -- keeps every train() call cheap
FINETUNE_EPOCHS = 2
FINETUNE_LR = 1e-4
FINETUNE_BATCH_SIZE = 4096
DEFAULT_MODEL_CONFIG = dict(n_bins=7, d_model=64, n_heads=4, n_layers=3, dropout=0.1)
# @crunch/keep:off

## Train & infer

These two functions are the actual submission entry points, called
`train()` once per walk-forward step and `infer()` once per moon of live
data. Everything above is just setup they rely on.

In [ ]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
) -> None:
    """Warm-start from the pretrained (or previously fine-tuned) checkpoint
    if one exists, fine-tune cheaply on the most recent moons, and persist
    the updated checkpoint for the next call / for infer()."""

    feature_columns = get_feature_columns(X_train)
    checkpoint_path = os.path.join(model_directory_path, CHECKPOINT_FILENAME)

    moons = np.sort(X_train["moon"].unique())
    recent_moons = moons[-FINETUNE_WINDOW_MOONS:]
    is_recent = X_train["moon"].isin(recent_moons).to_numpy()

    X_bins = X_train.loc[is_recent, feature_columns].to_numpy().astype(np.int64)
    target = y_train.loc[is_recent, "target"].to_numpy()

    if os.path.exists(checkpoint_path):
        print(f"Found existing checkpoint at {checkpoint_path} -- warm-starting fine-tune.")
        model, checkpoint = load_checkpoint(checkpoint_path)
        target_mean, target_std = checkpoint["target_mean"], checkpoint["target_std"]
        config = checkpoint["config"]
    else:
        print(
            f"WARNING: no checkpoint found at {checkpoint_path} -- training a fresh "
            "model from scratch on CPU. This should only happen if the pretrained "
            "resource upload didn't land in model_directory_path -- check the run log."
        )
        target_mean, target_std = float(target.mean()), float(target.std())
        config = {**DEFAULT_MODEL_CONFIG, "n_features": len(feature_columns)}
        model = build_model(config)

    y_scaled = (target - target_mean) / target_std

    dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(X_bins).long(), torch.from_numpy(y_scaled).float()
    )
    loader = torch.utils.data.DataLoader(dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(FINETUNE_EPOCHS):
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(yb)
        print(f"  fine-tune epoch {epoch + 1}/{FINETUNE_EPOCHS}  loss={total_loss / len(y_scaled):.4f}")

    os.makedirs(model_directory_path, exist_ok=True)
    save_checkpoint(checkpoint_path, model, config, feature_columns, target_mean, target_std)


def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
) -> pd.DataFrame:
    """Load the current checkpoint and score this moon of data."""

    checkpoint_path = os.path.join(model_directory_path, CHECKPOINT_FILENAME)
    model, checkpoint = load_checkpoint(checkpoint_path)
    feature_columns = checkpoint["feature_columns"]

    X_bins = torch.from_numpy(X_test[feature_columns].to_numpy().astype(np.int64)).long()

    model.eval()
    with torch.no_grad():
        pred_scaled = model(X_bins).numpy()

    predictions = X_test[["id", "moon"]].copy()
    # Spearman is scale/shift invariant, so the standardized prediction's
    # ranking is identical to an unstandardized one -- no need to invert.
    predictions["prediction"] = pred_scaled
    return predictions

## Load the data

In [ ]:
X_train, y_train, X_test = crunch_tools.load_data()
print(X_train.shape, y_train.shape, X_test.shape)

## Held-out validation — and a timing check

For this to exercise the real warm-start path (not a cold start), copy
your `transformer_checkpoint.pt` into a local `resources/` folder next to
this notebook before running this cell — that's the same default
directory `crunch_tools.test()` uses locally, matching what
`model_directory_path` will point to on the platform.

In [ ]:
%%time
LOCAL_MODEL_DIRECTORY = "resources"

feature_columns = get_feature_columns(X_train)
moons = np.sort(X_train["moon"].unique())
holdout_moons = moons[-50:]

is_holdout = X_train["moon"].isin(holdout_moons)
X_fit, y_fit = X_train.loc[~is_holdout], y_train.loc[~is_holdout]
X_val, y_val = X_train.loc[is_holdout], y_train.loc[is_holdout]

train(X_fit, y_fit, LOCAL_MODEL_DIRECTORY)
val_pred_df = infer(X_val, LOCAL_MODEL_DIRECTORY)

y_val_target = y_val["target"].to_numpy()
print(f"\nHoldout Spearman over last {len(holdout_moons)} moons: {spearman(y_val_target, val_pred_df['prediction'].to_numpy()):.4f}")

## Local test

`crunch_tools.test()` runs your `train()` / `infer()` exactly the way the
platform will, and checks the output format before you submit. Always run
this before pushing.

In [ ]:
crunch_tools.test()